# ☁️ Day 8: GCP + Your First Cloud Deployment

## Mission
Deploy your first Python app to Google Cloud Platform!

---

### Today's Topics
- **Part 1:** GCP Fundamentals (Regions, Zones, Projects)
- **Part 2:** Deploy to Cloud Run — Serverless! 🚀

## ☁️ GCP Basics

| Concept | What It Means |
|---------|---------------|
| **Project** | Container for all your GCP resources |
| **Region** | Geographic location (us-central1, asia-south1) |
| **Zone** | Specific data center within a region |
| **Cloud Run** | Serverless containers - pay only what you use |

---

## Step 1: Check gcloud CLI

In [7]:
# Check gcloud version and auth status
!gcloud --version


Google Cloud SDK 559.0.0
bq 2.1.28
core 2026.02.27
gcloud-crc32c 1.0.0
gsutil 5.35


In [9]:
# Check if authenticated
!gcloud auth list
! gcloud config set account 'krishna27sep2025@gmail.com'

                          Credentialed Accounts
ACTIVE  ACCOUNT
*       krishna27sep2025@gmail.com
        krishnavardhan3232@gmail.com
        vertex-ai-service-account@e2e-etl-project.iam.gserviceaccount.com

To set the active account, run:
    $ gcloud config set account `ACCOUNT`

Updated property [core/account].


In [10]:
# List your GCP projects
!gcloud projects list

PROJECT_ID                      NAME              PROJECT_NUMBER  ENVIRONMENT
api-1-487405                    api-1             897976377976
e2e-etl-project                 e2e-etl-project   279141399436
gen-lang-client-0127373536      openclaw-api      294475900129
oc-api-zindabad                 OC-api-zindabad   998196766929
og-ocp-etl                      OG-OCP-etl        746635318095
project-862c4514-44b6-4b33-aec  My First Project  699074662325


In [12]:
!gcloud config set project e2e-etl-project

Updated property [core/project].


In [13]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=sZiKGAybLySOyDdusaLrYW6jntqGhd&access_type=offline&code_challenge=gCDr6vAZc9yVHJoQAKAfQ_jQ9sFXPDt0g6iRP1vFonQ&code_challenge_method=S256


Credentials saved to file: [/Users/krishnavardhan/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "e2e-etl-project" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


---

## Step 2: Create Your Flask App

In [11]:
# Create the app directory
import os
app_dir = "/Users/krishnavardhan/projects/GCP_GENAI/gcp-genai-daily-grind/56_DAY_PRACTICE/day8_gcp/my_flask_app"
os.makedirs(app_dir, exist_ok=True)
print(f"Created: {app_dir}")

Created: /Users/krishnavardhan/projects/GCP_GENAI/gcp-genai-daily-grind/56_DAY_PRACTICE/day8_gcp/my_flask_app


In [16]:
# Create main.py - Flask application
main_py = '''
from flask import Flask, jsonify
import os

app = Flask(__name__)

@app.route('/')
def hello():
    return jsonify({
        "message": "Hello from Cloud Run!",
        "status": "deployed",
        "platform": "Google Cloud Run"
    })

@app.route('/health')
def health():
    return jsonify({"status": "healthy"})

@app.route('/info')
def info():
    return jsonify({
        "app": "Flask on Cloud Run",
        "port": os.environ.get("PORT", 8080)
    })

if __name__ == "__main__":
    port = int(os.environ.get("PORT", 8080))
    app.run(host="0.0.0.0", port=port)
'''

with open(f"{app_dir}/main.py", "w") as f:
    f.write(main_py)
print("Created: main.py")

Created: main.py


In [14]:
# Create requirements.txt
requirements_txt = """flask>=2.0.0
gunicorn>=20.0.0
"""

with open(f"{app_dir}/requirements.txt", "w") as f:
    f.write(requirements_txt)
print("Created: requirements.txt")

Created: requirements.txt


---

## Step 3: Create Dockerfile

In [17]:
# Create Dockerfile
dockerfile = '''# Use official Python runtime
FROM python:3.11-slim

# Set working directory
WORKDIR /app

# Copy requirements first (for caching)
COPY requirements.txt .

# Install dependencies
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY . .

# Expose the port Cloud Run expects
EXPOSE 8080

# Use gunicorn for production (handles multiple requests)
CMD ["gunicorn", "--bind", "0.0.0.0:8080", "main:app"]
'''

with open(f"{app_dir}/Dockerfile", "w") as f:
    f.write(dockerfile)
print("Created: Dockerfile")
print("\n📝 Dockerfile Contents:")
print(dockerfile)

Created: Dockerfile

📝 Dockerfile Contents:
# Use official Python runtime
FROM python:3.11-slim

# Set working directory
WORKDIR /app

# Copy requirements first (for caching)
COPY requirements.txt .

# Install dependencies
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY . .

# Expose the port Cloud Run expects
EXPOSE 8080

# Use gunicorn for production (handles multiple requests)
CMD ["gunicorn", "--bind", "0.0.0.0:8080", "main:app"]



---

## Step 4: Test Locally (Optional)

In [18]:
# Install flask and gunicorn locally to test
!pip install flask gunicorn -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [19]:
# Test the app locally
# In terminal, run: cd my_flask_app && python main.py
# Then visit http://localhost:8080

print("To test locally, run in terminal:")
print(f"cd {app_dir}")
print("python main.py")
print("\nThen visit: http://localhost:8080")

To test locally, run in terminal:
cd /Users/krishnavardhan/projects/GCP_GENAI/gcp-genai-daily-grind/56_DAY_PRACTICE/day8_gcp/my_flask_app
python main.py

Then visit: http://localhost:8080


---

## Step 5: Deploy to Cloud Run

In [20]:
# Set your project ID
# REPLACE with your GCP project ID
PROJECT_ID = "e2e-etl-project"  # <-- CHANGE THIS!

# Set the project
!gcloud config set project {PROJECT_ID}

Updated property [core/project].


In [21]:
# Enable Cloud Run API (if not already enabled)
!gcloud services enable run.googleapis.com

In [22]:
# Deploy to Cloud Run
# This command builds the container and deploys it
!cd {app_dir} && gcloud run deploy \
  --source . \
  --region us-central1 \
  --allow-unauthenticated \
  --platform managed

Service name (myflaskapp):  ^C


---

## 📊 How Cloud Run Works

```
User Request
     │
     ▼
┌─────────────────────────────────────┐
│         Cloud Load Balancer         │
└─────────────────────────────────────┘
     │
     ▼
┌─────────────────────────────────────┐
│           Cloud Run                 │
│  • Auto-scales 0 → 1000+ instances │
│  • Pay only for what you use        │
│  • No server management!            │
└─────────────────────────────────────┘
     │
     ▼
┌─────────────────────────────────────┐
│      Your Container (Docker)        │
│  • Flask app                        │
│  • gunicorn WSGI server             │
└─────────────────────────────────────┘
```

---

## 💰 Cloud Run Pricing (Important for Interviews!)

| Resource | Price |
|----------|-------|
| vCPU | $0.24 per 100,000 vCPU-seconds |
| Memory | $0.05 per 100,000 GiB-seconds |
| **Free Tier** | 180,000 vCPU-seconds + 360,000 GiB-seconds/month |
| **Idle = $0** | When no requests, you pay NOTHING! |

---

## 🎯 Interview Punch

> **"I deployed my first app to Cloud Run last week. It auto-scales from zero to thousands of requests, and I only pay per request. That's serverless containers — no infrastructure management needed."**

---

## ✅ Today's Checklist

- [ ] Install gcloud CLI ✅ (already installed!)
- [ ] Create GCP project (or use existing)
- [ ] Write simple Flask app ✅
- [ ] Create Dockerfile ✅
- [ ] Deploy to Cloud Run
- [ ] Test your live endpoint!
- [ ] **Tell someone: "I'm deployed on GCP!"**

---

*Mantra: Thaggedhe Le — Cloud is the future!* ☁️🚀